# Baseline — Base VLM on Google Colab (AUTHORITATIVE RUN)

**Model:** `Qwen/Qwen3-VL-4B-Instruct` — pretrained weights only.
**NO LoRA. NO training. NO fine-tuning.**

This is the official baseline experiment. It must complete and be saved
**before** any fine-tuning begins.

| Setting | Value |
|---|---|
| Evaluation set | 178 frozen held-out test images |
| Prompts | 3 frozen prompts |
| Generations | 178 x 3 = **534** |
| Track A (primary) | binary OK vs Defective, on the frozen 12 OK + 12 Defective slice |
| Track B (secondary) | 12-class defect type, full 178-image set |
| Decoding | greedy (`do_sample=False`) |
| Seed | 42 |

> **Runtime → Change runtime type → T4 GPU** before running.

> The defect classes in this dataset are **synthetic** (painted onto real OK
> castings by script). Results describe synthetic-defect detection, not real
> industrial defect recognition.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No CUDA GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
GPU_MEM_GB = round(props.total_memory / 1e9, 1)
print('GPU name       :', GPU_NAME)
print('GPU memory     :', GPU_MEM_GB, 'GB')
print('CUDA capability:', f'{props.major}.{props.minor}')
print('bf16 (native)  :', props.major >= 8,
      f'| torch reports {torch.cuda.is_bf16_supported()} '
      '(True on T4 but emulated -> we use fp16)')


## 2. Get the project — extract the ZIP to the Colab runtime disk

Google Drive is nearly full, so the project is **not** extracted into Drive.
Drive is mounted read-only in practice: we read one ZIP from it and unpack
into the Colab runtime disk at `/content/`, which is ephemeral and roomy.

Nothing is ever written back to Drive. Baseline results stay on the runtime
disk and are downloaded to your computer at the end (section 14).

> **The runtime disk is wiped when the session ends.** Download the results
> ZIP before closing Colab.


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = Path('/content/drive/MyDrive')
EXTRACT_TO = Path('/content')
PROJECT    = Path('/content/casting-defect-vlm')

# The project ZIP is identified by its CONTENTS, not its filename, so a
# rename (archive.zip, casting-defect-vlm-colab.zip, ...) does not matter.
PROJECT_MARKERS = [
    'config/config.yaml',
    'src/prompts.py',
    'scripts/run_baseline.py',
    'results/baseline/eval_subset.json',
]

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted (read-only use: we only read the ZIP from it)')
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    print('not in Colab — using local project at', PROJECT)


### 2.0 Find the project ZIP — by content, not by filename

Scans Drive for ZIPs and picks the one that actually contains the project.
This also catches the easy mix-up of uploading the **Kaggle dataset** ZIP,
which holds the images but none of the code, config, or frozen evaluation
subsets — the baseline cannot be reproduced from it.


In [ ]:
def _zip_names(zp, limit=5000):
    """Entry names inside a ZIP, with a single top-level folder stripped."""
    try:
        with zipfile.ZipFile(zp) as zf:
            names = zf.namelist()[:limit]
    except Exception as e:
        return None, None
    tops = {n.split('/')[0] for n in names if '/' in n}
    prefix = (next(iter(tops)) + '/') if len(tops) == 1 else ''
    stripped = {n[len(prefix):] if prefix and n.startswith(prefix) else n
                for n in names}
    return stripped, prefix.rstrip('/')

def classify_zip(zp):
    """Return (kind, matched_markers, top_level_prefix)."""
    names, prefix = _zip_names(zp)
    if names is None:
        return 'unreadable', [], None
    hits = [m for m in PROJECT_MARKERS if m in names]
    if len(hits) == len(PROJECT_MARKERS):
        return 'project', hits, prefix
    if any('defect_dataset/' in n for n in names):
        return 'dataset', hits, prefix
    return 'unknown', hits, prefix

ZIP_IN_DRIVE = None
if IN_COLAB:
    candidates = sorted(DRIVE_ROOT.glob('*.zip')) + sorted(DRIVE_ROOT.glob('*/*.zip'))
    assert candidates, f'No .zip files found in {DRIVE_ROOT}.'

    print(f'Scanning {len(candidates)} ZIP file(s) in Drive:\n')
    report = []
    for zp in candidates:
        kind, hits, prefix = classify_zip(zp)
        report.append((zp, kind, hits, prefix))
        extra = '' if kind == 'project' else f'  ({len(hits)}/{len(PROJECT_MARKERS)} markers)'
        print(f'  {zp.name:<42} {zp.stat().st_size/1e6:>7.0f} MB   -> {kind.upper()}{extra}')

    projects = [r for r in report if r[1] == 'project']
    datasets = [r for r in report if r[1] == 'dataset']

    if not projects:
        lines = ['', 'NO PROJECT ZIP FOUND IN DRIVE.', '']
        if datasets:
            lines += [
                f'{datasets[0][0].name} is the Kaggle DATASET zip, not the project.',
                'It contains the 1,200 images but NOT the code, config, or the',
                'frozen evaluation subsets, so the baseline cannot run from it.', '']
        lines += ['Upload the PROJECT zip (~118 MB) built on your machine at',
                  '  ~/casting-defect-vlm-colab.zip',
                  'to the root of My Drive. It must contain:']
        lines += [f'    {m}' for m in PROJECT_MARKERS]
        raise SystemExit('\n'.join(lines))

    ZIP_IN_DRIVE = projects[0][0]
    print(f'\nUsing project ZIP: {ZIP_IN_DRIVE}')
    if projects[0][3]:
        print(f'  top-level folder inside: {projects[0][3]}/')
    if len(projects) > 1:
        print(f'  NOTE: {len(projects)} project ZIPs found; using the first.')


### 2.1 Storage check

Refuses to continue if the runtime disk cannot hold the project plus the
~9 GB model cache.


In [ ]:
def _free_gb(path):
    """Free space in GB on the filesystem containing `path`."""
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize / 1e9

def _total_gb(path):
    st = os.statvfs(path)
    return st.f_blocks * st.f_frsize / 1e9

MODEL_CACHE_GB = 9.0          # Qwen3-VL-4B safetensors
EXTRACTED_GB   = 0.13         # measured: ~125 MB unzipped
HEADROOM_GB    = 3.0

if IN_COLAB:
    assert ZIP_IN_DRIVE is not None and ZIP_IN_DRIVE.exists(), \
        'No project ZIP resolved in section 2.0 — re-run that cell.'

    zip_gb   = ZIP_IN_DRIVE.stat().st_size / 1e9
    drive_free = _free_gb('/content/drive/MyDrive')
    disk_free  = _free_gb('/content')

    # Estimate the unpacked size from the ZIP's own directory listing.
    with zipfile.ZipFile(ZIP_IN_DRIVE) as zf:
        est_gb = sum(i.file_size for i in zf.infolist()) / 1e9
        n_entries = len(zf.infolist())

    needed = est_gb + MODEL_CACHE_GB + HEADROOM_GB

    print('=' * 62)
    print('STORAGE CHECK')
    print('=' * 62)
    print(f'Google Drive free space      : {drive_free:.2f} GB')
    print(f'Colab runtime free space     : {disk_free:.2f} GB '
          f'(of {_total_gb("/content"):.0f} GB)')
    print(f'Project ZIP size             : {zip_gb*1000:.0f} MB ({n_entries} entries)')
    print(f'Estimated extracted size     : {est_gb*1000:.0f} MB')
    print(f'Model cache needed           : {MODEL_CACHE_GB:.1f} GB')
    print(f'Headroom                     : {HEADROOM_GB:.1f} GB')
    print(f'Total required on /content   : {needed:.2f} GB')
    print('=' * 62)

    if disk_free < needed:
        raise SystemExit(
            f'INSUFFICIENT COLAB DISK SPACE: {disk_free:.2f} GB free but '
            f'{needed:.2f} GB required.\n'
            'Runtime -> Disconnect and delete runtime, then start a fresh T4 '
            'session and re-run. Do not proceed — the model download would '
            'fail part-way through.')
    print(f'OK — {disk_free:.1f} GB free, {needed:.1f} GB required.')

    # Drive is only read from, so low Drive space is not fatal. Note it anyway.
    if drive_free < 0.5:
        print(f'\nNOTE: Drive has only {drive_free:.2f} GB free. That is fine — '
              'nothing is written back to Drive.')
else:
    print(f'local run — /content checks skipped; free space here: '
          f'{_free_gb(str(PROJECT)):.1f} GB')


### 2.2 Extract to the runtime disk


In [ ]:
if IN_COLAB:
    if PROJECT.exists():
        print(f'{PROJECT} already exists — removing for a clean extract')
        shutil.rmtree(PROJECT)

    print(f'extracting {ZIP_IN_DRIVE.name} -> {EXTRACT_TO} ...')
    with zipfile.ZipFile(ZIP_IN_DRIVE) as zf:
        zf.extractall(EXTRACT_TO)

    # The ZIP may or may not wrap everything in a top-level folder.
    if not PROJECT.exists():
        candidates = [d for d in EXTRACT_TO.iterdir()
                      if d.is_dir() and (d / 'config' / 'config.yaml').exists()]
        assert candidates, (
            f'Extracted, but no project folder found under {EXTRACT_TO}. '
            f'Contents: {[d.name for d in EXTRACT_TO.iterdir()][:20]}')
        PROJECT = candidates[0]
        print(f'project root detected as {PROJECT}')

    n_files = sum(1 for _ in PROJECT.rglob('*') if _.is_file())
    size_mb = sum(f.stat().st_size for f in PROJECT.rglob('*') if f.is_file()) / 1e6
    print(f'extracted {n_files} files, {size_mb:.0f} MB')

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('PROJECT :', PROJECT)
print('cwd     :', Path.cwd())


### 2.3 Verify the project structure and the frozen inputs

The split and evaluation subsets were frozen locally. The baseline is only
comparable to the later fine-tuned run if these arrived byte-identical, so
their SHA-256 prefixes are printed for you to check against the local copies.


In [ ]:
import hashlib, json

# --- required directories ---
for d in ['config', 'src', 'scripts', 'data', 'results']:
    p = PROJECT / d
    assert p.is_dir(), f'MISSING DIRECTORY: {d}/'
    print(f'  OK  {d + "/":<16} ({sum(1 for _ in p.rglob("*") if _.is_file())} files)')

# --- required files, with hashes ---
print()
required = [
    'config/config.yaml',
    'data/processed/test.jsonl',
    'data/splits/splits.csv',
    'results/baseline/eval_subset.json',
    'results/baseline/balanced_subset.json',
    'src/prompts.py',
    'src/inference.py',
    'scripts/__init__.py',
    'scripts/run_baseline.py',
    'scripts/generate_baseline_report.py',
]
for f in required:
    p = PROJECT / f
    assert p.exists(), f'MISSING FILE: {f} — the ZIP is incomplete'
    print(f'  OK  {f:<44} sha256={hashlib.sha256(p.read_bytes()).hexdigest()[:16]}')

# --- frozen evaluation subsets ---
sub = json.loads((PROJECT / 'results/baseline/eval_subset.json').read_text())
bal = json.loads((PROJECT / 'results/baseline/balanced_subset.json').read_text())
print()
print('frozen eval set :', sub['n'], 'images')
print('balanced slice  :', bal['n'], f"({bal['n_ok']} OK / {bal['n_defective']} Defective)",
      '| seed', bal['seed'], '|', bal['n_source_groups'], 'source groups')

# --- the images themselves ---
IMG_ROOT = PROJECT / 'data/raw/output/defect_dataset'
assert IMG_ROOT.is_dir(), f'MISSING IMAGE ROOT: {IMG_ROOT}'
n_images = len(list(IMG_ROOT.rglob('*.jpg')))
print(f'\nimage root      : {IMG_ROOT}')
print(f'images on disk  : {n_images}')
assert n_images == 1200, f'expected 1200 dataset images, found {n_images}'

missing = [r for r in sub['relpaths'] if not (IMG_ROOT / r).exists()]
assert not missing, f'{len(missing)} evaluation images missing, e.g. {missing[:3]}'
print(f'all {len(sub["relpaths"])} frozen evaluation images present')

# --- results/baseline must not already hold results ---
stale = [f.name for f in (PROJECT / 'results/baseline').iterdir()
         if f.name not in {'.gitkeep', 'eval_subset.json', 'balanced_subset.json'}]
if stale:
    print(f'\nNOTE: results/baseline already contains {stale} — these will be overwritten.')
else:
    print('\nresults/baseline holds only the frozen subsets — no prior run to overwrite.')


## 3. Install dependencies

Colab ships a recent torch — we do **not** override it. Only the libraries
the project needs on top are installed.


In [ ]:
!pip install -q 'transformers>=5.0.0' 'accelerate>=1.0.0' 'peft>=0.14.0' 'pyyaml>=6.0' 'scikit-learn>=1.4' 'pandas>=2.0' 'pillow>=10.0' 'matplotlib>=3.8' 'scipy>=1.11'
print('installed')


In [ ]:
import torch, transformers, platform
print('python      :', platform.python_version())
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
print('Qwen3VLForConditionalGeneration: available')


> If the import above fails, `transformers` is too old for Qwen3-VL. Run
> `!pip install -q git+https://github.com/huggingface/transformers` and
> **restart the runtime**, then re-run from section 2.

> **If anything in this notebook errors, stop and report the exact error.**
> Do not work around it by changing the model, the prompts, the evaluation
> set, or the decoding settings — that would silently break comparability.


## 4. The frozen prompts

Printed in full so the run is self-documenting. These are used **unchanged**
for the fine-tuned evaluation. They deliberately do not list the dataset's
12 class names — that would hand the base model the answer space.


In [ ]:
from src.prompts import EVAL_PROMPTS
import hashlib

for pid, text in sorted(EVAL_PROMPTS.items()):
    h = hashlib.sha256(text.encode()).hexdigest()[:16]
    print('=' * 66)
    print(f'{pid}   sha256={h}')
    print('=' * 66)
    print(text)
    print()


## 5. Load the model (base, NO adapter)

dtype is chosen from the GPU: bf16 where supported (A100/L4), fp16 otherwise
(T4). This is a hardware detail and is recorded in `run_metadata.json`.


In [ ]:
import time
from src.utils import load_config

cfg = load_config()

# Pick the dtype the GPU actually supports.
# torch.cuda.is_bf16_supported() returns True on a T4, but Turing (sm_75)
# only EMULATES bf16 — it is markedly slower there. Native bf16 needs
# Ampere (sm_80) or newer, so gate on compute capability instead.
_cc = torch.cuda.get_device_capability(0)
use_bf16 = _cc[0] >= 8
print(f'compute capability: {_cc[0]}.{_cc[1]} '
      f'(native bf16 requires 8.0+)')
DTYPE = 'bfloat16' if use_bf16 else 'float16'
print('selected dtype:', DTYPE, '(bf16 supported:', use_bf16, ')')

from src.inference import load_model

t0 = time.time()
loaded = load_model(
    model_name=cfg['model']['model_name'],
    adapter_path=None,          # <-- BASE MODEL. No LoRA. This is the point.
    dtype=DTYPE,
    attn_implementation=cfg['model']['attn_implementation'],
    min_pixels=cfg['model'].get('min_pixels'),
    max_pixels=cfg['model'].get('max_pixels'),
)
MODEL_LOAD_SECONDS = time.time() - t0

print(f'model loading time : {MODEL_LOAD_SECONDS:.1f} s')
print('device             :', loaded.model.device)
print('dtype              :', next(loaded.model.parameters()).dtype)
print('adapter            :', loaded.adapter_path, '(None = base model, correct)')
print('VRAM allocated     :', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')
assert loaded.adapter_path is None, 'BASELINE MUST USE THE BASE MODEL'


## 6. Timed smoke test (2 generations)

Measures the real per-generation cost and projects the full run before
committing to all 534.


In [ ]:
from src.prepare_data import load_jsonl
from src.baseline import select_eval_subset
from src.prompts import parse_prediction
from src.inference import generate

test_examples = load_jsonl(cfg['data']['test_file'])
subset = select_eval_subset(test_examples, n=cfg['baseline']['eval_subset_size'],
                            seed=cfg['project']['seed'], reuse=True)
N_IMAGES  = len(subset)
N_PROMPTS = len(cfg['baseline']['prompt_ids'])
N_GEN     = N_IMAGES * N_PROMPTS
print('frozen evaluation set:', N_IMAGES, 'images x', N_PROMPTS, 'prompts =', N_GEN, 'generations')

# Time one generation per prompt on 2 images = 6 timed samples,
# so the projection reflects all three prompt lengths, not just prompt_1.
times = []
for ex in subset[:2]:
    for pid in cfg['baseline']['prompt_ids']:
        t = time.time()
        out = generate(loaded, ex['image'], prompt_id=pid,
                       max_new_tokens=cfg['model']['max_new_tokens'],
                       do_sample=cfg['inference']['do_sample'])
        dt = time.time() - t
        times.append(dt)
        print(f"\n--- {ex['relpath']} | truth={ex['ground_truth']} | {pid} | {dt:.1f}s ---")
        print(out)
        print('parsed ->', parse_prediction(out))

PER_GEN = sum(times) / len(times)
PROJECTED_MIN = PER_GEN * N_GEN / 60

print('\n' + '=' * 62)
print('SMOKE TEST REPORT')
print('=' * 62)
print(f'GPU name                     : {GPU_NAME}')
print(f'GPU VRAM                     : {GPU_MEM_GB} GB')
print(f'VRAM currently allocated     : {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'Model                        : {cfg["model"]["model_name"]}')
print(f'dtype                        : {next(loaded.model.parameters()).dtype}')
print(f'Adapter (must be None)       : {loaded.adapter_path}')
print(f'Model loading time           : {MODEL_LOAD_SECONDS:.1f} s')
print(f'Timed samples                : {len(times)} ({len(times)//N_PROMPTS} images x {N_PROMPTS} prompts)')
print(f'Generation time per image    : {PER_GEN:.2f} s  '
      f'(min {min(times):.2f}, max {max(times):.2f})')
print(f'Total generations to run     : {N_GEN}')
print(f'PROJECTED TOTAL RUNTIME      : {PROJECTED_MIN:.0f} min '
      f'({PROJECTED_MIN/60:.1f} hours)')
print('=' * 62)
print()
print('STOP HERE. Review the projected runtime before approving the full run.')


## 7. Run the full baseline

534 generations. Raw responses are streamed to JSONL as they are produced,
so an interrupted Colab session never loses completed work.

This writes `baseline_results.csv`, `baseline_raw_outputs.jsonl`,
`run_metadata.json`, the metrics JSON, confusion matrices and the error CSV.


## 6b. APPROVAL GATE — the full run does not start automatically

The next cell will **not** run until you explicitly approve it. This exists
so that a "Run all" cannot kick off 534 generations before you have looked
at the projected runtime above.

To approve: change `APPROVED_FOR_FULL_RUN` to `True` in the cell below,
then run it and continue.


In [ ]:
# ---------------------------------------------------------------
# Set to True ONLY after reviewing the smoke-test projection above.
APPROVED_FOR_FULL_RUN = False
# ---------------------------------------------------------------

if not APPROVED_FOR_FULL_RUN:
    print('FULL RUN NOT APPROVED — stopping here by design.')
    print(f'Projected runtime was {PROJECTED_MIN:.0f} min for {N_GEN} generations.')
    print('Set APPROVED_FOR_FULL_RUN = True above, re-run this cell, then continue.')
else:
    print(f'Approved. Proceeding with {N_GEN} generations '
          f'(~{PROJECTED_MIN:.0f} min projected).')


> Runs **in this process**, reusing the model already loaded above.
> Calling `!python scripts/run_baseline.py` here instead would load a
> second ~8 GB copy of the weights while the first is still resident —
> about 16 GB on a 15 GB T4, i.e. a guaranteed OOM.


In [ ]:
assert APPROVED_FOR_FULL_RUN, (
    'Full run not approved. Review the smoke-test projection, then set '
    'APPROVED_FOR_FULL_RUN = True in the cell above.')

from scripts.run_baseline import run_baseline, print_two_track_summary

t0 = time.time()
res_df, payload, meta = run_baseline(cfg, loaded=loaded)   # loaded = BASE model
TOTAL_INFERENCE_SECONDS = time.time() - t0

print_two_track_summary(payload, cfg, meta['n_images'], len(res_df),
                        title='BASELINE RESULTS (base model, NOT fine-tuned)')
print(f'\ntotal wall time: {TOTAL_INFERENCE_SECONDS/60:.1f} min')


### Run summary


In [ ]:
import pandas as pd

res = pd.read_csv('results/baseline/baseline_results.csv')
meta = json.loads(Path('results/baseline/run_metadata.json').read_text())

print('=' * 66)
print('BASELINE RUN SUMMARY')
print('=' * 66)
print(f"Model name            : {meta['model_id']}")
print(f"Model class           : {meta['model_class']}")
print(f"Fine-tuned            : {meta['fine_tuned']}   (must be False)")
print(f"Adapter               : {meta['adapter']}   (must be None)")
print(f"GPU name              : {GPU_NAME}")
print(f"GPU memory            : {GPU_MEM_GB} GB")
print(f"Device                : {meta['device']}")
print(f"dtype                 : {meta['dtype']}")
print(f"Model loading time    : {MODEL_LOAD_SECONDS:.1f} s")
print(f"Number of images      : {meta['n_images']}")
print(f"Number of prompts     : {meta['n_prompts']}")
print(f"Total generations     : {meta['total_generations']}")
print(f"Rows written          : {len(res)}")
print(f"Total inference time  : {meta['total_inference_seconds']/60:.1f} min")
print(f"Average generation    : {meta['avg_generation_seconds']:.2f} s")
print(f"transformers          : {meta['transformers_version']}")
print(f"torch                 : {meta['torch_version']}")
print(f"python                : {meta['python_version']}")
print(f"Decoding              : {meta['generation']}")
print(f"Seed                  : {meta['seed']}")
print(f"Prompt hashes         : {meta['prompt_sha256_16']}")
print('=' * 66)

assert meta['fine_tuned'] is False and meta['adapter'] is None, \
    'This is not a base-model run!'

# Fold in the notebook-measured model load time.
meta['model_load_seconds'] = round(MODEL_LOAD_SECONDS, 2)
Path('results/baseline/run_metadata.json').write_text(json.dumps(meta, indent=2))
print('run_metadata.json updated with model_load_seconds')


## 8. Verify the run is complete and valid

Fails loudly rather than letting a partial run be treated as the baseline.


In [ ]:
expected = len(subset) * len(cfg['baseline']['prompt_ids'])
assert len(res) == expected, f'expected {expected} rows, got {len(res)}'

pairs = set(zip(res['relpath'], res['prompt_id']))
assert len(pairs) == expected, 'duplicate or missing (image, prompt) pairs'

assert set(res['relpath']) == set(sub['relpaths']), \
    'evaluated images differ from the frozen subset'

errs = res['generation_error'].notna().sum()
print(f'rows                 : {len(res)} / {expected}')
print(f'unique image-prompt  : {len(pairs)}')
print(f'generation errors    : {errs}')
print(f'unparseable          : {(res.parsed_prediction == "Unparseable").sum()}')
print()
print('BASELINE VALID — the same image/prompt pairs are reusable for the',
      'fine-tuned model.' if errs == 0 else 'NOTE: some generations errored; see generation_error.')


## 9. TRACK A (PRIMARY) — binary OK vs Defective


In [ ]:
m = json.loads(Path('results/baseline/baseline_metrics.json').read_text())
a = m['track_a_binary']
bal_m, full_m = a['balanced_subset'], a['full_test_set']

pd.DataFrame([
    {'view': 'balanced slice (12 OK + 12 Def)', **{k: bal_m.get(k) for k in
     ['accuracy','balanced_accuracy','precision_defective','recall_defective',
      'f1_defective','macro_f1','ok_recall','defective_recall']}},
    {'view': 'full test set (178)', **{k: full_m.get(k) for k in
     ['accuracy','balanced_accuracy','precision_defective','recall_defective',
      'f1_defective','macro_f1','ok_recall','defective_recall']}},
]).set_index('view').T


In [ ]:
for name, d in [('BALANCED SLICE', bal_m), ('FULL TEST SET', full_m)]:
    cm = d['confusion_matrix']
    print(name)
    print('  TN (OK->OK)              :', cm['true_negative_ok_as_ok'])
    print('  FP (OK->Defective)       :', cm['false_positive_ok_as_defective'])
    print('  FN (Defective->OK)       :', cm['false_negative_defective_as_ok'])
    print('  TP (Defective->Defective):', cm['true_positive_defective_as_defective'])
    print('  OK recall       :', d['ok_recall'], '95% CI', d['ok_recall_95ci'])
    print('  Defective recall:', d['defective_recall'], '95% CI', d['defective_recall_95ci'])
    print()
print(bal_m.get('small_sample_warning'))
print(bal_m.get('ci_basis'))


In [ ]:
print(full_m['classification_report_text'])


In [ ]:
from IPython.display import Image, display
for f in ['baseline_confusion_matrix_balanced.png', 'baseline_confusion_matrix.png']:
    p = Path('results/baseline') / f
    if p.exists():
        print(f); display(Image(str(p)))


## 10. TRACK B (SECONDARY) — 12-class defect type

**Read coverage first.** The prompts never showed the model the 12-label
vocabulary, so many responses will not map to a class. Metrics cover only
those that do — a self-selected, optimistically biased subset.


In [ ]:
b = m['track_b_defect_type']['full_test_set']
print('coverage        :', b.get('coverage'),
      f"({b.get('n_valid_class_predictions')}/{b.get('n_total')} valid)")
print('unmatched       :', b.get('n_unmatched'))
for k in ['accuracy','macro_precision','macro_recall','macro_f1',
          'weighted_precision','weighted_recall','weighted_f1']:
    print(f'{k:<18}:', b.get(k))
print()
print(b.get('classification_report_text', 'n/a'))


In [ ]:
p = Path('results/baseline/baseline_defect_type_confusion_matrix.png')
display(Image(str(p))) if p.exists() else print('not generated')


## 11. Raw outputs

Every response preserved verbatim, including unparseable ones — nothing is
discarded.


In [ ]:
with open('results/baseline/baseline_raw_outputs.jsonl') as fh:
    raws = [json.loads(l) for l in fh]
print('raw records:', len(raws), '\n')
for r in raws[:6]:
    print('=' * 72)
    print(r['relpath'], '| truth:', r['ground_truth'],
          '| type:', r['true_defect_type'], '| prompt:', r['prompt_id'])
    print(r['raw_response'])
    print('-> parsed:', r['parsed_prediction'], '| class:', r.get('predicted_class'),
          '| conf:', r.get('confidence'), '| correct:', r['correct'])


## 12. Error analysis and required examples


In [ ]:
print(res['error_type'].value_counts(), '\n')
ea = pd.read_csv('results/baseline/error_analysis.csv')
print(ea['failure_flags'].value_counts())


### Said OK for a Defective casting (false negatives — missed defects)


In [ ]:
fn = res[(res.ground_truth=='Defective') & (res.parsed_prediction=='OK')]
print(len(fn), 'false negatives')
fn[['relpath','true_defect_type','prompt_id','confidence','raw_response']].head(8)


### Said Defective for an OK casting (false positives — false alarms)


In [ ]:
fp = res[(res.ground_truth=='OK') & (res.parsed_prediction=='Defective')]
print(len(fp), 'false positives')
fp[['relpath','prompt_id','predicted_defect_type','confidence','raw_response']].head(8)


### Hallucinated a defect type on an OK casting


In [ ]:
h = res[(res.ground_truth=='OK') &
        (~res.predicted_defect_type.astype(str).isin(['None','Unknown','nan']))]
print(len(h), 'hallucinated defect types')
h[['relpath','prompt_id','predicted_defect_type','predicted_class','confidence']].head(8)


### Correct predictions


In [ ]:
res[res.correct][['relpath','ground_truth','prompt_id','parsed_prediction','confidence']].head(8)


## 13. Generate the baseline report


In [ ]:
!python scripts/generate_baseline_report.py
print(Path('reports/baseline_report.md').read_text()[:3000])


## 14. Package and download the baseline results

Results stay on the **runtime disk** — nothing is written back to Drive,
which is nearly full. The ZIP below contains only `results/baseline/`
(a few MB) and is downloaded straight to your computer.

> **Do this before the runtime disconnects.** `/content` is wiped when the
> session ends and the results would be lost.


In [ ]:
import shutil

RESULTS_DIR = PROJECT / 'results' / 'baseline'
assert RESULTS_DIR.is_dir(), f'no results at {RESULTS_DIR}'

print('Contents of results/baseline/:')
total = 0
for f in sorted(RESULTS_DIR.iterdir()):
    if f.is_file():
        total += f.stat().st_size
        print(f'  {f.name:<50}{f.stat().st_size:>12,} bytes')
print(f'  {"TOTAL":<50}{total:>12,} bytes ({total/1e6:.1f} MB)')

# Sanity-check the run completed before packaging it.
required = ['baseline_results.csv', 'baseline_raw_outputs.jsonl',
            'run_metadata.json', 'baseline_metrics.json', 'error_analysis.csv']
missing = [f for f in required if not (RESULTS_DIR / f).exists()]
assert not missing, (
    f'Baseline incomplete — missing {missing}. Do NOT treat this as a '
    'finished baseline; re-run section 7.')
print('\nall required artefacts present')

# ZIP contains ONLY results/baseline/ — no dataset, no code, no model.
ZIP_OUT = '/content/baseline_results'
archive = shutil.make_archive(ZIP_OUT, 'zip', root_dir=str(RESULTS_DIR))
print(f'\nwrote {archive} ({Path(archive).stat().st_size/1e6:.1f} MB)')
print('NOTE: nothing was written to Google Drive.')


### Download to your computer

If the automatic download is blocked by your browser, open the Colab file
browser (folder icon, left sidebar), find `/content/baseline_results.zip`,
and download it manually.


In [ ]:
if IN_COLAB:
    from google.colab import files
    try:
        files.download('/content/baseline_results.zip')
        print('download started — check your browser')
    except Exception as e:
        print('automatic download failed:', e)
        print('Use the file browser (folder icon) -> /content/baseline_results.zip')
else:
    print('not in Colab — results are already on disk at', RESULTS_DIR)


## 15. STOP HERE

The baseline is complete. **Do not start LoRA training in this notebook.**

Before fine-tuning:
1. Confirm section 8's validation passed and `baseline_results.csv` has 534 rows.
2. Unzip `baseline_results.zip` into your **local** project at
   `results/baseline/`, then run `python scripts/generate_baseline_report.py`.
3. Review the numbers.

```bash
cd ~/casting-defect-vlm
unzip -o ~/Downloads/baseline_results.zip -d results/baseline/
.venv/bin/python scripts/generate_baseline_report.py
```

Fine-tuning is `notebooks/04_finetuning.ipynb`, and `scripts/train_model.py`
refuses a full training run until baseline results exist.

> Confidence values are **model-reported, not calibrated probabilities**.
> Track B measures recognition of **synthetic** defect textures, not real
> industrial defect recognition.
